
Se desarrolló un script para **homologar el video proporcionado por CENTRO** mediante homografía, obteniendo una vista aérea del campo de juego.  
La homografía se aplicó seleccionando las **cuatro esquinas del campo en orden, siguiendo el sentido de las manecillas del reloj**, lo que permitió transformar la perspectiva original a una vista cenital uniforme.  

A partir de esta transformación se extrajeron **402 frames**, los cuales fueron posteriormente **etiquetados en Roboflow** para construir el dataset de entrenamiento.


In [ ]:
import cv2
import numpy as np
import os

# Ruta del video
VIDEO_PATH = "IMG_9933.MOV"
OUTPUT_DIR = "imagenes_homograficas"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Tamaño destino (ejemplo: 640x480)
W, H = 640, 480

# Lista para guardar puntos
pts_src = []
scale = 0.4  # factor de reduccion para mostrar la imagen

def click_event(event, x, y, flags, param):
    global pts_src
    if event == cv2.EVENT_LBUTTONDOWN:
        # Escalar coordenadas de vuelta al tamaño original
        pts_src.append([int(x/scale), int(y/scale)])
        cv2.circle(frame_small, (x, y), 5, (0, 0, 255), -1)
        cv2.imshow("Selecciona 4 puntos", frame_small)

# Abrir video
cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame = cap.read()
if not ret:
    print("No se pudo abrir el video")
    exit()

# Reducir imagen para seleccion
frame_small = cv2.resize(frame, (int(frame.shape[1]*scale), int(frame.shape[0]*scale)))
cv2.imshow("Selecciona 4 puntos", frame_small)
cv2.setMouseCallback("Selecciona 4 puntos", click_event)

print("Haz clic en las 4 esquinas del campo (en orden).")
cv2.waitKey(0)
cv2.destroyAllWindows()

if len(pts_src) != 4:
    print("Debes seleccionar exactamente 4 puntos.")
    exit()

pts_src = np.array(pts_src, dtype=np.float32)
pts_dst = np.array([[0,0],[W,0],[W,H],[0,H]], dtype=np.float32)

# Calcular homografía
H_matrix, _ = cv2.findHomography(pts_src, pts_dst)

# Extraer frames cada 2 segundos
fps = int(cap.get(cv2.CAP_PROP_FPS))
interval_seconds = 2   # cada 2 segundos
interval = int(fps * interval_seconds)

frame_id = 0
saved = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_id % interval == 0:
        warped = cv2.warpPerspective(frame, H_matrix, (W,H))
        cv2.imwrite(os.path.join(OUTPUT_DIR, f"frame_{saved:04d}.jpg"), warped)
        saved += 1
    frame_id += 1

cap.release()
print(f"✅ Se guardaron {saved} frames en {OUTPUT_DIR}")


Haz clic en las 4 esquinas del campo (en orden).
✅ Se guardaron 402 frames en imagenes_homograficas
